# GT-Centric Cell Tracking Viewer & Exporter Pipeline (S2.3)

このノートブックは、**Ground Truth (GT) を100%基準**とした細胞トラッキング可視化データ (`viewer_data.json`) を抽出・生成し、3軸 MIP 射影画像と共に GitHub Pages (`kito2718/kaggle_Biohub-Cell_Tracking_During_Development2` の `gh-pages` ブランチ) へ自動デプロイするための統合処理ノートブックです。

---

## プロジェクト構成

本ノートブックは Kaggle Notebook環境専用の構成で動作します。

### Kaggle Notebook環境の構成
```text
/kaggle/
├── working/                                    # 作業ディレクトリ (カレントディレクトリ)
│   └── s2_03_gt_html_viewer.ipynb              # 本実行ノートブック
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       ├── train/                         # 訓練用データセット (.zarr / .geff)
    │       │   ├── xxxx.zarr/
    │       │   └── xxxx.geff/
    │       └── test/                          # 提出用データセット (.zarr)
    └── datasets/
        └── aaaa1597/
            ├── zarr-offline-installation-wheels/ # オフラインインストール用 zarr Wheels
            ├── tracksdata-wheels/                # オフラインインストール用 tracksdata Wheels (*.whl)
            ├── kaggle-cell-tracking-competition/  # 評価・処理用ソースコード (src/)
            └── btc-s106-progress/             # 継続実行・途中再開(Resume)用Dataset
```

## コミット対象構造 (`gh-pages` ブランチ)
```text
https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2 (branch: gh-pages)
 ├ index.html
 └ viewer_data/
 　 ├ datasets.json
 　 ├ 44b6_f28707c6/
 　 │ ├ mips/
 　 │ │ ├ frame_000_xy.png
 　 │ │ ├ frame_000_xz.png
 　 │ │ ├ frame_000_yz.png
 　 │ │ └ ...
 　 │ └ viewer_data.json
 　 └ 44b6_12dfb391/ ...
```
### 必要な Kaggle Datasets & Add-ons (Secrets) の事前準備手順

本ノートブックを Kaggle 環境で安定して連続実行・自動デプロイするために、以下の Kaggle Datasets および Secrets の設定を行ってください。

1. **必要な Kaggle Datasets の追加 (+ Add Data)**:
   - **`zarr-offline-installation-wheels`** (`/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels`):
     - インターネット接続オフの環境で `zarr` をインストールするためのオフライン Wheel 群データセット。
   - **`tracksdata-wheels`** (`/kaggle/input/datasets/aaaa1597/tracksdata-wheels`):
     - インターネット接続オフの環境で `tracksdata`, `geff`, `btrack` 等をインストールするためのオフライン Wheel 群データセット (`*.whl`)。
   - **`kaggle-cell-tracking-competition`** (`/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition`):
     - 公式評価指標および `tracking_cellmot` / `tracksdata` モジュール群が含まれるソースコードデータセット (`src/`)。
   - **`btc-s106-progress`** (`/kaggle/input/datasets/aaaa1597/btc-s106-progress`):
     - 9時間セッション制限対策の継続実行・途中再開 (Resume) 用 Dataset (`progress.json` や過去のチェックポイントを保持)。

2. **Add-ons > Secrets の設定**:
   - `GITHUB_TOKEN`: GitHub への可視化データ自動同期・プッシュに必要な GitHub Personal Access Token。
   - `KAGGLE_USERNAME`: Kaggle ユーザー名 (`aaaa1597`)。
   - `KAGGLE_KEY`: Kaggle API Token Key。

---

## 処理フローチャート (Pipeline Flowchart)

パイプライン全体の一括処理、途中再開 (Resume) 判定、GT 100%ノード抽出 (degree=0 の孤立ノードを含む)、および GitHub Pages への `--depth 1` 高速デプロイの処理フローです。枠内の名称はノートブックに実装されている実際の関数名およびセル番号と対応しています。

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell2["Cell 2: オフライン パッケージライブラリインストール<br>(zarr, tracksdata, btrack, geff)"]
    Cell2 --> Cell3["Cell 3: 環境パラメータ設定 & モジュールインポート"]
    Cell3 --> Cell4["Cell 4: check_environment()<br>(動作環境 Fail-Fast チェック)"]
    class Cell4 func;
    Cell4 --> Cell6["Cell 6: get_dataset_pairs()<br>(.zarr & .geff ペア探索)"]
    class Cell6 func;

    Cell6 --> Cell12_Init["Cell 12: process_all_datasets()<br>(全データセットバッチ処理開始)"]
    class Cell12_Init func;

    subgraph Cell12_Loop ["Cell 12: 全データセットバッチ処理ループ"]
        LoopStart{"データセットループ開始"}
        class LoopStart loop;

        LoopStart --> CheckSkip{"CONTINUOUS_FLAG == True <br>&& 処理完了済みデータセット?"}
        class CheckSkip cond;

        CheckSkip -- Yes (スキップ) --> LoopEndDummy[ ]
        style LoopEndDummy fill:none,stroke:none,width:0px,height:0px;

        CheckSkip -- No --> StepGT["Cell 7: export_gt_viewer_data()<br>1. GT 100%全載せ抽出<br>2. degree=0 孤立GTノードカウント<br>3. TP/FP/FN & グローバル統計算出"]
        class StepGT func;

        StepGT --> StepMIP["Cell 9: generate_pred_graph() & ensure_mip_images()<br>3軸 MIP 射影画像 (xy/xz/yz) の配置・生成"]
        class StepMIP func;

        StepMIP --> SaveCheck["Cell 11: save_completed_dataset()<br>進捗チェックポイントの更新"]
        class SaveCheck func;

        SaveCheck --> LoopEndDummy
    end

    Cell12_Loop --> Cell14["Cell 14: ensure_index_html()<br>HTMLビューアー テンプレート生成"]
    class Cell14 func;

    Cell14 --> Cell15["Cell 15: push_to_github_pages()<br>1. git clone --branch gh-pages --depth 1 (高速浅いクローン)<br>2. index.html, viewer_data.json, mips/, datasets.json 同期<br>3. remote repo へ自動 push"]
    class Cell15 func;

    Cell15 --> Cell16["Cell 16: メインエントリーポイント実行"]
    class Cell16 func;

    Cell16 --> End([処理完了])
```


In [ ]:
# Cell 2: オフライン パッケージライブラリインストール
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels rustworkx bidict ilpy imagecodecs polars btrack zarr Pillow
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels geff geff-spec
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels tracksdata


In [ ]:
# Cell 3: 環境・実行パラメータ設定 & モジュールインポート (CONFIGURATION & IMPORTS)
import os
import sys
import glob
import time
import json
import shutil
import tempfile
import numpy as np
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32
import zarr
from pathlib import Path
from PIL import Image

# ==============================================================================
# Cell 3: 環境・実行パラメータ設定 & モジュールインポート (CONFIGURATION & IMPORTS)
# ==============================================================================
# 1. 途中再開 (Resume) & チェックポイント設定
CONTINUOUS_FLAG = True       # True: 自動チェックポイント保存 & スキップを有効化
RESET_CHECKPOINT = False     # True: 過去のチェックポイントを一度クリアして一からスタート

# 2. GitHub Pages 自動デプロイ設定
PUSH_TO_GITHUB = True        # True: 処理結果を GitHub Pages へ自動反映
GITHUB_REPO = 'https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2.git'
BRANCH_NAME = 'gh-pages'
from kaggle_secrets import UserSecretsClient
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN") if PUSH_TO_GITHUB else ''

# 3. データセットパス設定
DATASET_SLUG = "btc-s106-progress"
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
WORKING_DIR = Path('/kaggle/working')

# 途中保存用 Progress Dataset ディレクトリの存在チェック
CHECKPOINT_DATASET_DIR = Path(f'/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}')
if not CHECKPOINT_DATASET_DIR.exists():
    raise FileNotFoundError(f"Checkpoint dataset directory not found: {CHECKPOINT_DATASET_DIR}")

CHECKPOINT_DATASET_PATH = CHECKPOINT_DATASET_DIR / 'gt_viewer_data.json'

# 4. パス設定 & モジュールインポート
KAGGLE_SRC_DIR = '/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition/src'
if not os.path.exists(KAGGLE_SRC_DIR):
    raise FileNotFoundError(f"Required Kaggle source directory not found: {KAGGLE_SRC_DIR}")

if KAGGLE_SRC_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_SRC_DIR)

# Ground Truth Evaluation & Metrics imports
import geff
import tracksdata as td
from tracksdata.graph import IndexedRXGraph
import tracking_cellmot.io
from tracking_cellmot.metrics import evaluate
from tracking_cellmot.io import open_dataset

print(f"Pipeline parameters initialized. CONTINUOUS_FLAG={CONTINUOUS_FLAG}, RESET_CHECKPOINT={RESET_CHECKPOINT}")
print("All required tracking_cellmot & tracksdata modules imported successfully.")

# === GPUなし環境での RuntimeError 対策 (CPUモンキーパッチ) ===
import torch
if not torch.cuda.is_available():
    def patched_process_on_gpu(
        image, tracks, scale, device,
        resample=False, target_scale=None,
        normalize=True, gamma=1.0,
        q_min=0.01, q_max=0.99, subsample_factor=1000,
        precomputed_quantiles=None
    ):
        torch_device = torch.device(device)
        image = image.astype(np.float32, copy=False)
        q1, q2 = None, None
        if normalize:
            q1 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_min)
            q2 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_max)
            if q1 is None or q2 is None:
                flat = image.ravel()[::subsample_factor]
                q1, q2 = np.quantile(flat, [q_min, q_max]).astype(np.float32)
            else:
                q1 = np.float32(q1)
                q2 = np.float32(q2)
        tensor = torch.from_numpy(image)
        tensor = tensor.to(torch_device, non_blocking=True)
        if normalize:
            tensor = (tensor - float(q1)) / (float(q2) - float(q1) + 1e-6)
            tensor = tensor.clamp(min=0.0)
            if gamma != 1.0:
                tensor = tensor.pow(gamma)
            tensor = tensor.clamp(0.0, 4.0)
        if resample:
            scale_arr = np.array(scale)
            target_scale_val = scale_arr.min() if target_scale is None else np.array(target_scale)
            zoom_factors = scale_arr / target_scale_val
            new_spatial_shape = (np.array(tensor.shape[1:]) * zoom_factors).astype(int).tolist()
            tensor = tensor[:, None]
            tensor = torch.nn.functional.interpolate(
                tensor, size=new_spatial_shape, mode="trilinear", align_corners=False
            )
            tensor = tensor[:, 0]
            if tracks is not None:
                import tracksdata as td
                import polars as pl
                node_attrs = tracks.node_attrs()
                orig_dtypes = {col: node_attrs.schema[col] for col in ["z", "y", "x"]}
                node_attrs = node_attrs.with_columns(
                    (pl.col("z") * zoom_factors[0]).round(0).cast(orig_dtypes["z"]),
                    (pl.col("y") * zoom_factors[1]).round(0).cast(orig_dtypes["y"]),
                    (pl.col("x") * zoom_factors[2]).round(0).cast(orig_dtypes["x"]),
                )
                tracks.update_node_attrs(
                    attrs=node_attrs.select("z", "y", "x").to_dict(),
                    node_ids=node_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),
                )
            scale = target_scale if target_scale is not None else (float(target_scale_val),) * 3
        return tensor, tracks, scale
    
    tracking_cellmot.io._process_on_gpu = patched_process_on_gpu
    print("All required tracking_cellmot & tracksdata modules imported successfully. CPU Monkey Patch applied.")

print(f"Pipeline parameters initialized. CONTINUOUS_FLAG={CONTINUOUS_FLAG}, RESET_CHECKPOINT={RESET_CHECKPOINT}")


In [ ]:
# Cell 4: 動作環境 Fail-Fast チェック関数
def check_environment():
    """
    環境要件(ライブラリ、データディレクトリ、GitHub Secret設定等)を即座に確認するFail-Fastチェック関数。
    iterdir() を用いてファイル存在を高速スキャンします。
    """
    print("Checking environment requirements...")
    
    # 1. 必須コアライブラリ・評価ライブラリのインポートチェック
    import zarr
    import polars as pl
    import geff
    import tracksdata as td
    import tracking_cellmot.io
    from tracking_cellmot.metrics import evaluate
    from tracking_cellmot.io import open_dataset
    print("  - [OK] Required libraries (zarr, polars, geff, tracksdata, tracking_cellmot) are verified.")

    # 2. データディレクトリの存在チェック
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")
    
    # 3. Zarr データセットの存在チェック
    zarr_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.zarr')])
    if not zarr_files:
        raise FileNotFoundError(f"No Zarr datasets found in {DATA_DIR}")
    print(f"  - [OK] Zarr target datasets found in '{DATA_DIR}' (Count: {len(zarr_files)}).")

    # 4. GEFF Ground Truth の存在チェック
    geff_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.geff')])
    if not geff_files:
        raise FileNotFoundError(f"No GEFF ground truth files found in {DATA_DIR}")
    print(f"  - [OK] GEFF ground truth files found in '{DATA_DIR}' (Count: {len(geff_files)}).")

    # 5. GitHub 自動同期設定のチェック
    if PUSH_TO_GITHUB:
        if not GITHUB_REPO or len(GITHUB_REPO.strip()) == 0:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_REPO is invalid or empty.")

        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e_sec:
            raise ValueError(f"PUSH_TO_GITHUB is True, but failed to fetch GITHUB_TOKEN from Secrets: {e_sec}") from e_sec
        
        if not token:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_TOKEN is not set in Secrets or env.")
        print(f"  - [OK] GitHub Auto-Push configuration (repo: '{GITHUB_REPO}') verified.")

    # 6. チェックポイント自動同期設定のチェック (CONTINUOUS_FLAG=True の時)
    if CONTINUOUS_FLAG:
        if not DATASET_SLUG:
            raise ValueError("CONTINUOUS_FLAG is True, but DATASET_SLUG is not defined.")
        try:
            from kaggle_secrets import UserSecretsClient
            u = UserSecretsClient().get_secret("KAGGLE_USERNAME")
            k = UserSecretsClient().get_secret("KAGGLE_KEY")
            if not u or not k:
                raise ValueError("KAGGLE_USERNAME or KAGGLE_KEY in Secrets is empty.")
        except Exception as e_k:
            raise ValueError(f"CONTINUOUS_FLAG is True, but failed to fetch Kaggle API secrets (KAGGLE_USERNAME / KAGGLE_KEY): {e_k}") from e_k
        print(f"  - [OK] Dataset Checkpoint Auto-Sync (slug: '{DATASET_SLUG}', secrets: verified) is valid.")

    print("Environment check PASSED successfully!")

check_environment()


## 2. Dataset Scanning & Ground Truth Loading

In [ ]:
# Cell 6: データセット対探索関数
def get_dataset_pairs(data_dir: Path):
    """
    DATA_DIR配下の Zarr と GEFF のペアを高速走査(iterdir)して取得する関数。
    """
    pairs = []
    zarr_paths = sorted([p for p in data_dir.iterdir() if p.name.endswith('.zarr')])
    for zarr_path in zarr_paths:
        dataset_name = zarr_path.stem
        geff_path = zarr_path.with_suffix('.geff')
        if geff_path.exists():
            pairs.append((dataset_name, zarr_path, geff_path))
    return pairs

dataset_pairs = get_dataset_pairs(DATA_DIR)
print(f"Discovered {len(dataset_pairs)} dataset pairs.")


In [ ]:
# Cell 7: 全 Dataset 統合 RAW CSV エクスポート (export_raw_csv_all) & GT 基準可視化データ抽出関数 (export_gt_viewer_data)
def export_raw_csv_all(data_dir: Path, out_dir: Path):
    """
    全 Dataset の生データ .geff から全ノード・エッジ属性テーブルを直接抽出し、
    先頭に dataset 列を付与して 2 つの統合 CSV (gt_nodes.csv, gt_edges.csv) にエクスポートする関数。
    出力先: out_dir / 'viewer_data' / 'gt_nodes.csv', 'gt_edges.csv'
    """
    import pandas as pd
    out_csv_dir = out_dir / 'viewer_data'
    out_csv_dir.mkdir(parents=True, exist_ok=True)
    
    all_nodes_dfs = []
    all_edges_dfs = []
    
    dataset_pairs = get_dataset_pairs(data_dir)
    print(f"  - [RAW CSV 統合抽出] 全 {len(dataset_pairs)} 件のデータセットを処理中...")
    
    for name, zarr_path, geff_path in dataset_pairs:
        try:
            ds_path = os.path.join(str(data_dir), name)
            ds = open_dataset(ds_path, normalize=True, require_tracks=True, device="cpu")
            gt_graph = ds.tracks
            
            nodes_df = gt_graph.node_attrs().to_pandas()
            edges_df = gt_graph.edge_attrs().to_pandas()
            
            nodes_df.insert(0, 'dataset', name)
            edges_df.insert(0, 'dataset', name)
            
            all_nodes_dfs.append(nodes_df)
            all_edges_dfs.append(edges_df)
        except Exception as e:
            print(f"  - [WARN] {name} の RAW CSV 抽出中に例外発生: {e}")

    if all_nodes_dfs:
        merged_nodes_df = pd.concat(all_nodes_dfs, ignore_index=True)
        merged_nodes_df.to_csv(out_csv_dir / 'gt_nodes.csv', index=False)
        print(f"  - [OK] viewer_data/gt_nodes.csv 出力完了: 全 {len(merged_nodes_df)} 行")

    if all_edges_dfs:
        merged_edges_df = pd.concat(all_edges_dfs, ignore_index=True)
        merged_edges_df.to_csv(out_csv_dir / 'gt_edges.csv', index=False)
        print(f"  - [OK] viewer_data/gt_edges.csv 出力完了: 全 {len(merged_edges_df)} 行")

def export_gt_viewer_data(gt_graph: IndexedRXGraph, pred_graph: IndexedRXGraph, dataset_name: str, out_dir: Path):
    """
    Ground Truth (GT) 基準の可視化データ (viewer_data.json) を抽出・出力する関数。
    孤立ノード (degree=0) および pred_edges_fp (誤検出予測エッジ) を抽出します。
    """
    eval_result = evaluate(pred_graph, gt_graph) if pred_graph is not None else None
    
    gt_nodes_tp = []
    gt_nodes_fn = []
    matched_gt_ids = set()
    gt_to_pred_map = {}
    
    if eval_result and hasattr(eval_result, 'node_matches'):
        for gt_id, pred_id in eval_result.node_matches.items():
            matched_gt_ids.add(int(gt_id))
            gt_to_pred_map[int(gt_id)] = int(pred_id)

    # GT グラフのノード・エッジ属性テーブル取得 (IndexedRXGraph 公式 API)
    gt_nodes_df = gt_graph.node_attrs().to_pandas()
    gt_edges_df = gt_graph.edge_attrs().to_pandas()
    
    connected_gt_ids = set(gt_edges_df['source_id']) | set(gt_edges_df['target_id'])
    isolated_gt_count = 0
    
    parent_map = {}
    if len(gt_edges_df) > 0:
        for row in gt_edges_df.itertuples(index=False):
            parent_map[int(row.target_id)] = int(row.source_id)

    for row in gt_nodes_df.itertuples(index=False):
        gt_id = int(row.node_id)
        t = int(row.t)
        z = float(row.z)
        y = float(row.y)
        x = float(row.x)
        parent_id = parent_map.get(gt_id, None)
        
        if gt_id not in connected_gt_ids:
            isolated_gt_count += 1
        
        if gt_id in matched_gt_ids:
            pred_id = int(gt_to_pred_map[gt_id])
            gt_nodes_tp.append([z, y, x, gt_id, pred_id, t, parent_id])
        else:
            gt_nodes_fn.append([z, y, x, gt_id, None, t, parent_id])

    pred_nodes_fp = []
    matched_pred_ids = set(gt_to_pred_map.values())
    
    if pred_graph is not None:
        pred_nodes_df = pred_graph.node_attrs().to_pandas()
        for row in pred_nodes_df.itertuples(index=False):
            pred_id = int(row.node_id)
            if pred_id not in matched_pred_ids:
                t = int(row.t)
                z = float(row.z)
                y = float(row.y)
                x = float(row.x)
                pred_nodes_fp.append([z, y, x, pred_id, None, t, None])

    gt_edges_tp = []
    gt_edges_fn = []
    matched_gt_edges = getattr(eval_result, 'edge_matches', {}) if eval_result else {}

    if len(gt_edges_df) > 0:
        for row in gt_edges_df.itertuples(index=False):
            u, v = int(row.source_id), int(row.target_id)
            if (u, v) in matched_gt_edges:
                gt_edges_tp.append([u, v])
            else:
                gt_edges_fn.append([u, v])

    # pred_edges_fp (誤検出予測エッジ) の抽出
    pred_edges_fp = []
    if pred_graph is not None:
        pred_edges_df = pred_graph.edge_attrs().to_pandas()
        matched_pred_edges_set = set(matched_gt_edges.values()) if matched_gt_edges else set()
        if len(pred_edges_df) > 0:
            for row in pred_edges_df.itertuples(index=False):
                pu, pv = int(row.source_id), int(row.target_id)
                if (pu, pv) not in matched_pred_edges_set:
                    pred_edges_fp.append([pu, pv])

    total_gt_nodes = len(gt_nodes_df)
    tp_count = len(gt_nodes_tp)
    fn_count = len(gt_nodes_fn)
    fp_count = len(pred_nodes_fp)
    
    precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0.0
    recall = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    payload = {
        'dataset': dataset_name,
        'global_summary': {
            'node_precision': precision,
            'node_recall': recall,
            'node_f1': f1,
            'edge_f1': getattr(eval_result, 'edge_f1', 0.0) if eval_result else 0.0,
            'gt_nodes_total': total_gt_nodes,
            'isolated_gt_nodes_count': isolated_gt_count,
            'node_tp_count': tp_count,
            'node_fn_count': fn_count,
            'node_fp_count': fp_count,
            'edge_tp_count': len(gt_edges_tp),
            'edge_fn_count': len(gt_edges_fn),
            'pred_edges_fp_count': len(pred_edges_fp)
        },
        'gt_nodes_tp': gt_nodes_tp,
        'gt_nodes_fn': gt_nodes_fn,
        'pred_nodes_fp': pred_nodes_fp,
        'gt_edges_tp': gt_edges_tp,
        'gt_edges_fn': gt_edges_fn,
        'pred_edges_fp': pred_edges_fp,
        'metadata': {
            'tooltips': {
                'gt_node_tp': 'GT細胞が存在し、予測細胞と正しくマッチしたノード (True Positive)',
                'gt_node_fn': 'GT細胞が存在するが、予測で検出漏れとなったノード (False Negative)',
                'pred_node_fp': 'GTが存在しない位置に誤って検出された予測ノード (False Positive)',
                'gt_edge_tp': 'GTの追跡リンクと正しく一致したトラッキングエッジ (Edge TP)',
                'gt_edge_fn': 'GTの追跡リンクが存在するが、予測で途切れたエッジ (Edge FN)',
                'pred_edges_fp': 'GTに存在しない誤った予測追跡リンク (Edge FP)',
                'isolated_gt_nodes': 'どのエッジとも接続していない孤立 GT 細胞ノード数 (100% 抽出保存対象)'
            }
        }
    }

    out_dataset_dir = out_dir / 'viewer_data' / dataset_name
    out_dataset_dir.mkdir(parents=True, exist_ok=True)
    
    with open(out_dataset_dir / 'viewer_data.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)

    print(f"  - [OK] viewer_data.json 出力完了 ({dataset_name}): GTノード={total_gt_nodes} (孤立={isolated_gt_count}), TP={tp_count}, FN={fn_count}, FP={fp_count}, pred_edges_fp={len(pred_edges_fp)}")
print("export_gt_viewer_data() defined.")

## 3. MIP Projection Image Generation

In [ ]:
def get_zarr_voxel_spacing(store) -> tuple:
    """
    OME-Zarr メタデータ (.attrs['multiscales']) から Z, Y, X 軸の物理ボクセルスケール (μm) を抽出する関数。
    """
    attrs = getattr(store, 'attrs', {})
    if 'multiscales' not in attrs:
        raise ValueError("不正な OME-Zarr メタデータ: '.attrs' 内に 'multiscales' が存在しません。")

    ms = attrs['multiscales']
    if not isinstance(ms, list) or len(ms) == 0:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales' 属性が空または不正です。")

    datasets = ms[0].get('datasets', [])
    if not datasets:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales[0].datasets' が空です。")

    transforms = datasets[0].get('coordinateTransformations', [])
    for t in transforms:
        if t.get('type') == 'scale':
            s = t.get('scale', [])
            if len(s) >= 4:
                scale_z, scale_y, scale_x = float(s[1]), float(s[2]), float(s[3])
                return scale_z, scale_y, scale_x
            elif len(s) == 3:
                scale_z, scale_y, scale_x = float(s[0]), float(s[1]), float(s[2])
                return scale_z, scale_y, scale_x

    raise ValueError("不正な OME-Zarr メタデータ: '.attrs' 内に有効な 'scale' 座標変換情報が見つかりません。")


# Cell 9: 予測グラフ生成 & 3軸 MIP 射影画像抽出関数 (generate_pred_graph & ensure_mip_images)
def generate_pred_graph(zarr_path: Path, max_search_radius_um: float = 7.0) -> IndexedRXGraph:
    """
    3D Zarr 画像ボリュームから細胞検出 (gaussian_filter + peak_local_max) および
    フレーム間リンキング (nearest neighbor / greedy) を行い、予測グラフ (IndexedRXGraph) を構築・返却する独立関数。
    今後モデルやアルゴリズムが改良・変更された場合も、この関数単位で容易に差し替えが可能です。
    """
    print(f"  - [予測グラフ生成] 3D細胞検出 & トラッキングを開始中 ({zarr_path.stem})...")
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)

    try:
        store = zarr.open(str(zarr_path), mode='r')
        scale_z, scale_y, scale_x = get_zarr_voxel_spacing(store)

        if hasattr(store, 'array_keys') or hasattr(store, 'keys'):
            keys = list(store.array_keys())
            if not keys:
                print(f"  - [WARN] Zarr グループ内にアレイキーが見つかりません: {zarr_path}")
                return pred_graph
            arr = store[keys[0]]
        else:
            arr = store

        shape = arr.shape
        if len(shape) == 5:
            data = arr[:, 0, :, :, :]
        elif len(shape) == 4:
            data = arr[:]
        elif len(shape) == 3:
            data = arr[np.newaxis, ...]
        else:
            print(f"  - [WARN] サポートされていない Zarr 形状 {shape}: {zarr_path.stem}")
            return pred_graph

        num_frames = data.shape[0]
        nodes = []
        global_node_id = 0
        from scipy.ndimage import gaussian_filter
        from skimage.feature import peak_local_max

        for t in range(num_frames):
            frame = data[t]
            img_min, img_max = frame.min(), frame.max()
            if img_max > img_min:
                img_norm = (frame.astype(np.float32, copy=False) - img_min) / (img_max - img_min)
            else:
                img_norm = np.zeros_like(frame, dtype=np.float32)

            img_smoothed = gaussian_filter(img_norm, sigma=2.0)
            peaks = peak_local_max(img_smoothed, min_distance=2, threshold_abs=0.12)

            if len(peaks) > 300:
                peak_intensities = [(img_smoothed[p[0], p[1], p[2]], p) for p in peaks]
                peak_intensities.sort(key=lambda x: x[0], reverse=True)
                peaks = [p for _, p in peak_intensities[:300]]

            for p in peaks:
                z, y, x = p
                phys_z, phys_y, phys_x = float(z * scale_z), float(y * scale_y), float(x * scale_x)
                nodes.append({
                    'node_id': global_node_id,
                    't': t,
                    'z': phys_z,
                    'y': phys_y,
                    'x': phys_x
                })
                pred_graph.add_node(
                    attrs={'t': int(t), 'z': phys_z, 'y': phys_y, 'x': phys_x},
                    index=int(global_node_id)
                )
                global_node_id += 1

        if not nodes:
            return pred_graph

        import pandas as pd
        from scipy.spatial.distance import cdist
        nodes_df = pd.DataFrame(nodes)

        frames = sorted(nodes_df['t'].unique())
        for idx, t in enumerate(frames[:-1]):
            t_next = frames[idx + 1]
            if t_next != t + 1:
                continue

            df_prev = nodes_df[nodes_df['t'] == t]
            df_curr = nodes_df[nodes_df['t'] == t_next]

            coords_prev = df_prev[['z', 'y', 'x']].values
            coords_curr = df_curr[['z', 'y', 'x']].values
            prev_ids = df_prev['node_id'].values
            curr_ids = df_curr['node_id'].values

            dists = cdist(coords_prev, coords_curr)
            used_curr = set()

            for i in range(len(df_prev)):
                min_idx = np.argmin(dists[i])
                min_dist = dists[i][min_idx]

                if min_dist <= max_search_radius_um and min_idx not in used_curr:
                    used_curr.add(min_idx)
                    src_id = int(prev_ids[i])
                    tgt_id = int(curr_ids[min_idx])
                    pred_graph.add_edge(src_id, tgt_id, attrs={})

        print(f"  - [OK] 予測グラフ生成完了 ({zarr_path.stem}): 予測ノード数={global_node_id}, 予測エッジ数={pred_graph.num_edges()}")
        return pred_graph
    except Exception as e:
        print(f"  - [WARN] 予測グラフ生成例外発生 ({zarr_path.stem}): {e}")
        return pred_graph

def ensure_mip_images(zarr_path: Path, dataset_name: str, out_dir: Path):
    """
    各フレームの 3軸 (XY, XZ, YZ) MIP 射影画像 (PNG) を生成する関数。
    Zarr の物理ボクセルスケール (μm) に基づいてアスペクト比を補正し、正方形キャンバスに描画・保存します。
    保存先: out_dir / 'viewer_data' / dataset_name / 'mips' / frame_[no]_[xy|xz|yz].png
    """
    dst_mips_dir = out_dir / 'viewer_data' / dataset_name / 'mips'
    dst_mips_dir.mkdir(parents=True, exist_ok=True)
    
    existing_mips = [p for p in dst_mips_dir.iterdir() if p.name.startswith('frame_') and p.name.endswith('.png')]
    if existing_mips:
        print(f"Skipping! MIP images already present for {dataset_name} ({len(existing_mips)} images).")
        return

    print(f"Extracting 3-axis MIP images for dataset: {dataset_name}...")
    try:
        store = zarr.open(str(zarr_path), mode='r')
        scale_z, scale_y, scale_x = get_zarr_voxel_spacing(store)

        if hasattr(store, 'array_keys') or hasattr(store, 'keys'):
            keys = list(store.array_keys())
            if not keys:
                print(f"Warning: No array keys found in Zarr group at {zarr_path}")
                return
            arr = store[keys[0]]
        else:
            arr = store

        shape = arr.shape
        if len(shape) == 5:
            data = arr[:, 0, :, :, :]
        elif len(shape) == 4:
            data = arr[:]
        elif len(shape) == 3:
            data = arr[np.newaxis, ...]
        else:
            print(f"Warning: Unsupported Zarr array shape {shape} for {dataset_name}")
            return

        num_frames = data.shape[0]
        start_time = time.time()
        
        for t in range(num_frames):
            # 進捗ログ出力 (最初、最後、および5フレーム毎)
            if t == 0 or t == num_frames - 1 or (t + 1) % 5 == 0:
                elapsed = time.time() - start_time
                pct = (t + 1) / num_frames * 100.0
                print(f"    [MIP進捗 {t+1}/{num_frames} ({pct:.1f}%)] 処理時間: {elapsed:.1f}秒")

            vol = data[t]  # (Z, Y, X)
            Z, Y, X = vol.shape
            
            # 物理空間サイズ (μm) の計算
            phys_z = Z * scale_z
            phys_y = Y * scale_y
            phys_x = X * scale_x

            # MIP (Maximum Intensity Projection) の算出
            mip_xy = np.max(vol, axis=0)  # (Y, X)
            mip_xz = np.max(vol, axis=1)  # (Z, X)
            mip_yz = np.max(vol, axis=2)  # (Z, Y)

            def to_norm_uint8(arr_2d):
                min_v, max_v = float(arr_2d.min()), float(arr_2d.max())
                if max_v > min_v:
                    return ((arr_2d - min_v) / (max_v - min_v) * 255.0).astype(np.uint8)
                return np.zeros_like(arr_2d, dtype=np.uint8)

            # Webビューアー画面での統一描画キャンバスサイズ (X, Y)
            target_w, target_h = X, Y

            # 1. XY MIP (Y, X) - 物理アスペクト比: (phys_x, phys_y)
            img_xy_raw = Image.fromarray(to_norm_uint8(mip_xy))
            img_xy = img_xy_raw.resize((target_w, target_h), Image.Resampling.BILINEAR)
            img_xy.save(dst_mips_dir / f"frame_{t:03d}_xy.png")

            # 2. XZ MIP (Z, X) - 物理幅: phys_x (μm), 物理高さ: phys_z (μm)
            calc_h_xz = max(1, int(target_h * (phys_z / max(phys_x, 1e-5))))
            img_xz_resized = Image.fromarray(to_norm_uint8(mip_xz)).resize((target_w, calc_h_xz), Image.Resampling.BILINEAR)
            canvas_xz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_xz, target_h)) // 2
            canvas_xz.paste(img_xz_resized.crop((0, 0, target_w, min(calc_h_xz, target_h))), (0, paste_y))
            canvas_xz.save(dst_mips_dir / f"frame_{t:03d}_xz.png")

            # 3. YZ MIP (Z, Y) - 物理幅: phys_y (μm), 物理高さ: phys_z (μm)
            calc_h_yz = max(1, int(target_h * (phys_z / max(phys_y, 1e-5))))
            img_yz_resized = Image.fromarray(to_norm_uint8(mip_yz)).resize((target_w, calc_h_yz), Image.Resampling.BILINEAR)
            canvas_yz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_yz, target_h)) // 2
            canvas_yz.paste(img_yz_resized.crop((0, 0, target_w, min(calc_h_yz, target_h))), (0, paste_y))
            canvas_yz.save(dst_mips_dir / f"frame_{t:03d}_yz.png")

        print(f"Successfully generated {num_frames * 3} MIP images for {dataset_name}.")
    except Exception as e:
        print(f"Error generating MIP images for {dataset_name}: {e}")

print("get_zarr_voxel_spacing(), ensure_mip_images() defined.")

## 4. Pipeline Execution & Resume Checkpoint

In [ ]:
# Cell 11: 途中再開 (Resume) チェックポイント関数 (load_completed_datasets & save_completed_dataset)
def load_completed_datasets(checkpoint_path: Path, continuous: bool, reset: bool):
    """CONTINUOUS_FLAG=True の場合に過去に処理完了したデータセット一覧をロードする関数。"""
    if reset or not continuous:
        return set()
    if checkpoint_path.exists():
        try:
            with open(checkpoint_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return set(data.get('completed', []))
        except Exception as e:
            print(f"Warning reading checkpoint {checkpoint_path}: {e}")
    return set()

def save_completed_dataset(checkpoint_path: Path, dataset_name: str, continuous: bool):
    """処理完了したデータセット名を進捗チェックポイント JSON (gt_viewer_data.json) へ更新保存する関数。"""
    if not continuous:
        return
    completed = load_completed_datasets(checkpoint_path, continuous=True, reset=False)
    completed.add(dataset_name)
    save_path = WORKING_DIR / 'gt_viewer_data.json'
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump({'completed': sorted(list(completed))}, f, indent=2)
    print(f"Updated checkpoint: {dataset_name} marked complete.")

print("load_completed_datasets(), save_completed_dataset() defined.")

In [ ]:
# Cell 12: 全データセット一括バッチ処理関数 (process_all_datasets)
def process_all_datasets():
    from tracking_cellmot.io import open_dataset
    dataset_pairs = get_dataset_pairs(DATA_DIR)
    total_datasets = len(dataset_pairs)
    
    if total_datasets == 0:
        print(f"  - [WARN] {DATA_DIR} 内に対象となる (.zarr, .geff) ペアが見つかりません。")
        return

    completed_datasets = load_completed_datasets(CHECKPOINT_DATASET_PATH, CONTINUOUS_FLAG, RESET_CHECKPOINT)
    print(f"Resuming pipeline. Already completed: {len(completed_datasets)} datasets.")

    for idx, (name, zarr_path, geff_path) in enumerate(dataset_pairs, 1):
        pct = idx / total_datasets * 100.0
        print(f"[データセット {idx}/{total_datasets} ({pct:.1f}%)] 処理中: {name}")
        
        if CONTINUOUS_FLAG and name in completed_datasets:
            print(f"Skipping already processed dataset: {name}")
            continue
        
        print(f"Processing dataset: {name}...")
        
        # 1. Ground Truth (GEFF) グラフの読み込み
        if not geff_path.exists():
            raise FileNotFoundError(f"Ground Truth GEFF file missing for dataset: {name} at {geff_path}")
        print(f"  - [GEFF読み込み] {geff_path.name} をパース中...")
        ds = open_dataset(os.path.join(str(DATA_DIR), name), normalize=True, require_tracks=True, device="cpu")
        gt_graph = ds.tracks

        # 2. GT 基準可視化データ (viewer_data.json) の抽出・出力
        pred_graph = generate_pred_graph(zarr_path)
        export_gt_viewer_data(gt_graph=gt_graph, pred_graph=pred_graph, dataset_name=name, out_dir=WORKING_DIR)

        # 3. 3軸 MIP 射影画像 (PNG) の生成・出力
        ensure_mip_images(zarr_path, name, WORKING_DIR)
        
        # 4. 進捗チェックポイントの更新
        save_completed_dataset(CHECKPOINT_DATASET_PATH, name, CONTINUOUS_FLAG)

    print("All dataset processing steps completed!")

process_all_datasets()


## 5. GitHub Pages Deployment (Shallow Clone --depth 1)

In [ ]:
# Cell 14: HTML ビューアーテンプレート生成関数 (ensure_index_html)
def ensure_index_html(out_dir: Path):
    """
    gh-pages ルートに配置される index.html テンプレートを生成する関数。
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    html_path = out_dir / 'index.html'
    html_content = """<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="UTF-8">
  <title>GT-Centric Cell Tracking Viewer</title>
      <style>
    body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 20px; background-color: #1a1a1a; color: #eee; }
    h1 { color: #4fc3f7; margin-top: 0; }
    .container { display: flex; flex-direction: column; gap: 20px; }
    .control-panel { background: #2a2a2a; padding: 15px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.3); }
    select, button { padding: 8px 12px; font-size: 14px; border-radius: 4px; background: #333; color: #fff; border: 1px solid #555; cursor: pointer; }
    .summary-card { background: #2a2a2a; padding: 15px; border-radius: 8px; margin-top: 10px; }
    .summary-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; }
    .metric-box { background: #333; padding: 12px; border-radius: 6px; border-left: 4px solid #4fc3f7; position: relative; }
    .metric-label { font-size: 12px; color: #aaa; text-transform: uppercase; display: flex; align-items: center; gap: 5px; }
    .metric-value { font-size: 20px; font-weight: bold; margin-top: 5px; color: #fff; }
    
    /* Tooltip styles */
    .tooltip-icon { display: inline-block; width: 16px; height: 16px; background: #555; color: #fff; border-radius: 50%; text-align: center; font-size: 11px; line-height: 16px; cursor: help; }
    .tooltip-container { position: relative; display: inline-block; }
    .tooltip-container .tooltip-text {
      visibility: hidden; width: 260px; background-color: #444; color: #fff; text-align: left;
      border-radius: 6px; padding: 8px 12px; position: absolute; z-index: 100; bottom: 125%; left: 50%;
      transform: translateX(-50%); opacity: 0; transition: opacity 0.3s; font-size: 12px; font-weight: normal;
      box-shadow: 0 4px 10px rgba(0,0,0,0.5); text-transform: none; line-height: 1.4;
    }
    .tooltip-container:hover .tooltip-text { visibility: visible; opacity: 1; }
    
    .legend-panel { display: flex; gap: 15px; flex-wrap: wrap; margin-top: 15px; background: #222; padding: 10px; border-radius: 6px; }
    .legend-item { display: flex; align-items: center; gap: 8px; font-size: 13px; }
    .color-badge { width: 12px; height: 12px; border-radius: 30%; display: inline-block; }
    .tp-color { background: #4caf50; }
    .fn-color { background: #f44336; }
    .fp-color { background: #ff9800; }
    .image-viewer { display: flex; gap: 15px; overflow-x: auto; padding: 10px 0; }
    .image-box { text-align: center; background: #222; padding: 10px; border-radius: 6px; }
    .image-box img { max-width: 300px; height: auto; border: 1px solid #444; }
      </style>
</head>
<body>
      <div class="container">
  <h1>GT-Centric Cell Tracking Viewer</h1>
    
    <div class="control-panel">
      <label for="dataset-select">Dataset: </label>
      <select id="dataset-select"><option value="">Loading datasets...</option></select>
    </div>

    <div class="summary-card">
      <h3>Global Summary Metrics</h3>
      <div class="summary-grid" id="metrics-grid">
        <!-- Dynamically populated metrics with tooltips -->
      </div>
      
      <div class="legend-panel" id="legend-panel">
        <!-- Dynamically populated legend items with tooltips -->
      </div>
    </div>

    <div class="image-viewer" id="mips-container">
      <!-- MIP images for current frame -->
    </div>
      </div>

      <script>
    async function init() {
      try {
        const res = await fetch('viewer_data/datasets.json');
        const data = await res.json();
        const select = document.getElementById('dataset-select');
        select.innerHTML = '';
        data.datasets.forEach(ds => {
          const opt = document.createElement('option');
          opt.value = ds;
          opt.textContent = ds;
          select.appendChild(opt);
        });
        if (data.datasets.length > 0) {
          loadDataset(data.datasets[0]);
        }
        select.addEventListener('change', (e) => loadDataset(e.target.value));
      } catch (e) {
        console.error('Failed to load datasets.json', e);
      }
    }

    async function loadDataset(datasetName) {
      try {
        const res = await fetch(`viewer_data/${datasetName}/viewer_data.json`);
        const payload = await res.json();
        renderSummary(payload);
        renderMIPs(datasetName, 0);
      } catch (e) {
        console.error(`Failed to load viewer data for ${datasetName}`, e);
      }
    }

    function renderSummary(payload) {
      const summary = payload.global_summary || {};
      const tooltips = payload.metadata?.tooltips || {};
      const grid = document.getElementById('metrics-grid');
      const legend = document.getElementById('legend-panel');

      grid.innerHTML = `
        <div class="metric-box">
          <div class="metric-label">Node Precision</div>
          <div class="metric-value">${(summary.node_precision || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Node Recall</div>
          <div class="metric-value">${(summary.node_recall || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Node F1</div>
          <div class="metric-value">${(summary.node_f1 || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">Edge F1</div>
          <div class="metric-value">${(summary.edge_f1 || 0).toFixed(4)}</div>
        </div>
        <div class="metric-box">
          <div class="metric-label">
            Isolated GT Nodes
            <span class="tooltip-container">
              <span class="tooltip-icon">i</span>
              <span class="tooltip-text">どのエッジとも接続していない孤立GT細胞ノード数 (100%抽出対象)</span>
            </span>
          </div>
          <div class="metric-value">${summary.isolated_gt_nodes_count || 0}</div>
        </div>
      `;

      legend.innerHTML = `
        <div class="legend-item">
          <span class="color-badge tp-color"></span>
          <span>GT Node TP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_node_tp || 'GT細胞と予測細胞が一致'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span class="color-badge fn-color"></span>
          <span>GT Node FN</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_node_fn || 'GT細胞が存在するが検出漏れ'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span class="color-badge fp-color"></span>
          <span>Pred Node FP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.pred_node_fp || '予測細胞が存在するがGTが存在しない'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span style="color:#4caf50">━━</span>
          <span>Edge TP</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_edge_tp || 'トラッキング成功エッジ'}</span>
          </span>
        </div>
        <div class="legend-item">
          <span style="color:#f44336">━━</span>
          <span>Edge FN</span>
          <span class="tooltip-container">
            <span class="tooltip-icon">i</span>
            <span class="tooltip-text">${tooltips.gt_edge_fn || '追跡途切れリンク'}</span>
          </span>
        </div>
      `;
    }

    function renderMIPs(datasetName, frameIdx) {
      const container = document.getElementById('mips-container');
      const frameStr = String(frameIdx).padStart(3, '0');
      container.innerHTML = `
        <div class="image-box">
          <div>XY Projection (Z-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_xy.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\\'http://www.w3.org/2000/svg\\' width=\\'200\\' height=\\'200\\'><rect width=\\'200\\' height=\\'200\\' fill=\\'%23333\\'/><text x=\\'50%\\' y=\\'50%\\' fill=\\'%23888\\' text-anchor=\\'middle\\'>No MIP Image</text></svg>'">
        </div>
        <div class="image-box">
          <div>XZ Projection (Y-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_xz.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\\'http://www.w3.org/2000/svg\\' width=\\'200\\' height=\\'200\\'><rect width=\\'200\\' height=\\'200\\' fill=\\'%23333\\'/><text x=\\'50%\\' y=\\'50%\\' fill=\\'%23888\\' text-anchor=\\'middle\\'>No MIP Image</text></svg>'">
        </div>
        <div class="image-box">
          <div>YZ Projection (X-MIP)</div>
          <img src="viewer_data/${datasetName}/mips/frame_${frameStr}_yz.png" onerror="this.src='data:image/svg+xml;utf8,<svg xmlns=\\'http://www.w3.org/2000/svg\\' width=\\'200\\' height=\\'200\\'/><text x=\\'50%\\' y=\\'50%\\' fill=\\'%23888\\' text-anchor=\\'middle\\'>No MIP Image</text></svg>'">
        </div>
      `;
    }

    init();
      </script>
</body>
</html>
"""
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
    print(f"Ensured index.html template at {html_path}")

print("ensure_index_html() defined.")


In [ ]:
# Cell 15: GitHub Pages 自動デプロイ関数 (push_to_github_pages)
def push_to_github_pages(working_dir: Path, repo_url: str, branch: str = 'gh-pages', token: str = '', enabled: bool = True):
    """
    GitHub Pages ブランチ (gh-pages) へ浅いクローン (--depth 1) を用いて index.html と viewer_data フォルダをデプロイ・プッシュする関数。
    """
    if not enabled:
        print("PUSH_TO_GITHUB is False. Skipping GitHub Pages deployment.")
        return

    print(f"[GitHub Deploy 3/5] GitHub Pages 自動同期・デプロイ開始")
    
    # 認証トークン確保
    from kaggle_secrets import UserSecretsClient
    auth_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not auth_token:
        raise ValueError("GITHUB_TOKEN is missing or empty. Cannot push to GitHub without authentication token.")

    authed_repo_url = repo_url.replace('https://', f'https://x-access-token:{auth_token}@')

    # Step 1: 送信対象データの検証
    print("[1/5] ローカルの可視化成果物 (viewer_data) をチェック中...")
    src_viewer_data = working_dir / 'viewer_data'
    if not src_viewer_data.exists():
        raise FileNotFoundError(f"【送信エラー】成果物フォルダ {src_viewer_data} が存在しません。")

    subdirs = [d for d in src_viewer_data.iterdir() if d.is_dir()]
    if not subdirs:
        raise FileNotFoundError(f"【送信エラー】{src_viewer_data} 内にデータセットフォルダが1つも存在しません。")

    total_mips = 0
    for ds_dir in subdirs:
        mips_dir = ds_dir / 'mips'
        if mips_dir.exists():
            total_mips += len([f for f in mips_dir.iterdir() if f.name.endswith('.png')])
            
    print(f"  - [OK] 検出データセット数: {len(subdirs)} 件, 検出 MIP 画像数: {total_mips} 枚")
    if total_mips == 0:
        print("  - [WARN] MIP 画像 (PNG) が 0 枚です。画像生成ステップを確認してください。")

    import subprocess
    import tempfile

    # Step 2: 一時作業用クローン
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_repo = Path(tmp_dir) / 'repo'
        print(f"[2/5] リポジトリ '{branch}' ブランチを浅いクローン (--depth 1) 中...")
        
        clone_res = subprocess.run(
            ["git", "clone", "--branch", branch, "--depth", "1", authed_repo_url, str(tmp_repo)],
            capture_output=True, text=True
        )
        if clone_res.returncode != 0:
            print(f"  - [新規作成] '{branch}' ブランチの取得に失敗したため初期化します: {clone_res.stderr.strip()}")
            init_res = subprocess.run(
                ["git", "clone", "--depth", "1", authed_repo_url, str(tmp_repo)],
                capture_output=True, text=True
            )
            if init_res.returncode != 0:
                raise RuntimeError(f"【Git Clone エラー】リポジトリのクローンに失敗しました:\n{init_res.stderr}")
            subprocess.run(["git", "checkout", "-b", branch], cwd=tmp_repo, check=True)

        # Step 3: ファイルの同期・配置
        print("[3/5] 成果物 (index.html, viewer_data/) をブランチへ同期・配置中...")
        ensure_index_html(working_dir)
        src_html = working_dir / 'index.html'
        if src_html.exists():
            shutil.copy2(src_html, tmp_repo / 'index.html')

        dst_viewer_data = tmp_repo / 'viewer_data'
        if dst_viewer_data.exists():
            shutil.rmtree(dst_viewer_data)
        shutil.copytree(src_viewer_data, dst_viewer_data)

        # datasets.json の生成
        datasets = [d.name for d in dst_viewer_data.iterdir() if d.is_dir()]
        with open(dst_viewer_data / 'datasets.json', 'w', encoding='utf-8') as f:
            json.dump({'datasets': sorted(datasets)}, f, indent=2)

        # Step 4: 差分確認とコミット・プッシュ (Python subprocess 完結)
        print("[4/5] Git の変更差分を確認中...")
        subprocess.run(["git", "config", "user.name", "Kaggle-Bot"], cwd=tmp_repo, check=True)
        subprocess.run(["git", "config", "user.email", "bot@kaggle.com"], cwd=tmp_repo, check=True)
        
        # -A フラグで新規追加・更新・削除(mips/*.png含む)を全ステージング
        subprocess.run(["git", "add", "-A"], cwd=tmp_repo, check=True)

        status_res = subprocess.run(["git", "status", "--porcelain"], cwd=tmp_repo, capture_output=True, text=True)
        changes = status_res.stdout.strip()
        
        if not changes:
            print("  - [OK] リモートブランチに変更差分はありません。Push を完了とみなします。")
            return

        print(f"  - 検出された差分件数: {len(changes.splitlines())} 件 (mips画像含む)")

        print("[5/5] GitHub リモートへコミット & プッシュを実行中...")
        commit_res = subprocess.run(
            ["git", "commit", "-m", "Auto-update GT-centric cell tracking viewer with MIP images"],
            cwd=tmp_repo, capture_output=True, text=True
        )
        if commit_res.returncode != 0:
            raise RuntimeError(f"【Git Commit エラー】コミットに失敗しました:\n{commit_res.stderr}")

        push_res = subprocess.run(
            ["git", "push", authed_repo_url, f"{branch}:{branch}"],
            cwd=tmp_repo, capture_output=True, text=True
        )
        if push_res.returncode != 0:
            raise RuntimeError(f"【Git Push エラー】GitHub リモートへの Push に失敗しました:\n{push_res.stderr}")

        print(f"\n[GitHub Deploy 完了] 成果物の GitHub Pages デプロイが成功しました！")
        print(f"  -> 公開URL: https://kito2718.github.io/kaggle_Biohub-Cell_Tracking_During_Development2/")


In [ ]:
# Cell 16: メイン実行エントリーポイント (メインパイプライン呼び出し)
push_to_github_pages(WORKING_DIR, GITHUB_REPO, BRANCH_NAME, GITHUB_TOKEN, PUSH_TO_GITHUB)


## 6. Pipeline Completed

GT-centric cell tracking visualization dataset export and GitHub Pages deployment successfully finished.

In [ ]:
# Cell 17: 全 Dataset 統合 RAW CSV エクスポート・高度 HTML ビューアー更新 & デプロイ専用関数 (run_raw_csv_and_deploy_viewer)
# ==============================================================================
# 本セルは Run All で自動実行されないよう全体が関数化されています。
# 実行する場合はセル末尾の # run_raw_csv_and_deploy_viewer(...) のコメントアウトを解除して実行してください。
# ==============================================================================
def run_raw_csv_and_deploy_viewer(export_csv=True, push_github=True):
    """
    全 Dataset の生データ .geff から統合 RAW CSV (gt_nodes.csv, gt_edges.csv) をエクスポートし、
    最新の高度 HTML ビューアーを出力して GitHub Pages へデプロイする一元実行関数。
    """
    # 1. 環境・デプロイパラメータ設定
    github_repo = 'https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2.git'
    branch_name = 'gh-pages'
    
    try:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN") if push_github else ''
    except Exception:
        github_token = ''
    
    print(f"[run_raw_csv_and_deploy_viewer 実行開始] export_csv={export_csv}, push_github={push_github}")
    
    # 2. 全 Dataset 統合 RAW CSV エクスポートの実行 (viewer_data/gt_nodes.csv & gt_edges.csv)
    if export_csv:
        print("\n--- 全 Dataset の生データ (.geff) からの統合 RAW CSV エクスポートを開始中 ---")
        export_raw_csv_all(DATA_DIR, WORKING_DIR)
    
    # 3. 最新の高度 HTML ビューアー (index.html) テンプレートの生成・配置
    print("\n--- 高度インタラクティブ HTML ビューアー (index.html) を Cell 17 内から生成・配置中 ---")
    html_path = WORKING_DIR / 'index.html'
    advanced_html_content = """<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="UTF-8">
  <title>GT-Centric Cell Tracking Viewer (Advanced S2.3)</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #121212; color: #e0e0e0; font-size: 12px; }
    .top-bar { background: #1e1e1e; padding: 8px 15px; display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #333; }
    h1 { font-size: 16px; color: #4fc3f7; font-weight: 600; margin: 0; }
    .selector-container { display: flex; align-items: center; gap: 10px; }
    input[type="text"], select, input[type="number"] { padding: 4px 8px; font-size: 12px; border-radius: 4px; background: #2a2a2a; color: #fff; border: 1px solid #444; }
    input[type="text"]:focus, select:focus { border-color: #4fc3f7; outline: none; }
    .summary-bar { display: flex; gap: 10px; padding: 8px 15px; background: #181818; border-bottom: 1px solid #2a2a2a; flex-wrap: wrap; }
    .card { background: #222; border-radius: 6px; padding: 6px 10px; border-left: 3px solid #4fc3f7; flex: 1; min-width: 130px; position: relative; }
    .card-title { font-size: 10px; color: #aaa; text-transform: uppercase; display: flex; align-items: center; justify-content: space-between; }
    .card-val { font-size: 15px; font-weight: bold; color: #fff; margin-top: 2px; }
    .tooltip-icon { display: inline-block; width: 13px; height: 13px; background: #444; color: #fff; border-radius: 50%; text-align: center; font-size: 9px; line-height: 13px; cursor: help; }
    .tooltip-icon:hover::after {
      content: attr(data-tooltip); position: absolute; z-index: 1000; bottom: 120%; left: 10px;
      width: 200px; background: #333; color: #fff; padding: 6px; border-radius: 4px; font-size: 10px;
      font-weight: normal; text-transform: none; box-shadow: 0 4px 8px rgba(0,0,0,0.5); border: 1px solid #555;
    }
    .player-bar { display: flex; align-items: center; gap: 12px; padding: 6px 15px; background: #1a1a1a; border-bottom: 1px solid #2a2a2a; }
    .btn { padding: 4px 10px; font-size: 12px; background: #333; color: #fff; border: 1px solid #555; border-radius: 4px; cursor: pointer; }
    .btn:hover { background: #444; border-color: #4fc3f7; }
    .slider { flex: 1; cursor: pointer; }
    .frame-badge { font-weight: bold; color: #4fc3f7; min-width: 90px; }
    .main-grid { display: grid; grid-template-columns: 1fr 300px; gap: 10px; padding: 10px; height: calc(100vh - 140px); overflow: hidden; }
    .left-panel { display: flex; flex-direction: column; gap: 10px; overflow-y: auto; padding-right: 5px; }
    .right-panel { display: flex; flex-direction: column; gap: 10px; overflow-y: auto; }
    .plot-section { background: #1e1e1e; border-radius: 6px; padding: 8px; border: 1px solid #2a2a2a; }
    .section-header { font-weight: 600; font-size: 12px; color: #4fc3f7; margin-bottom: 6px; display: flex; justify-content: space-between; align-items: center; }
    .plots-3d-row { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; height: 360px; }
    .plotly-box { width: 100%; height: 100%; border-radius: 4px; background: #141414; }
    .mips-2d-row { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; height: 340px; }
    .mip-canvas-box { position: relative; width: 100%; height: 100%; background: #000; border-radius: 4px; overflow: hidden; border: 1px solid #333; }
    .mip-canvas-box canvas { width: 100%; height: 100%; display: block; cursor: crosshair; }
    .mip-title { position: absolute; top: 6px; left: 8px; font-size: 11px; font-weight: bold; color: #fff; background: rgba(0,0,0,0.6); padding: 2px 6px; border-radius: 3px; z-index: 10; }
    .gizmo-overlay { position: absolute; bottom: 8px; right: 8px; width: 45px; height: 45px; pointer-events: none; z-index: 10; }
    .inspector-card { background: #1e1e1e; border-radius: 6px; padding: 12px; border: 1px solid #333; }
    .inspector-table { width: 100%; border-collapse: collapse; margin-top: 8px; }
    .inspector-table td { padding: 4px 6px; border-bottom: 1px solid #2a2a2a; font-size: 11px; }
    .inspector-table td.label { color: #888; width: 40%; }
    .inspector-table td.val { color: #fff; font-weight: 600; word-break: break-all; }
    .badge { padding: 2px 6px; border-radius: 3px; font-size: 10px; font-weight: bold; color: #fff; }
    .bg-tp { background: #2e7d32; }
    .bg-fn { background: #c62828; }
    .bg-fp { background: #ef6c00; }
  </style>
</head>
<body>
  <div class="top-bar">
    <h1>GT-Centric Cell Tracking Viewer</h1>
    <div class="selector-container">
      <label>Search:</label>
      <input type="text" id="dataset-search" placeholder="e.g. no1, no.1, 44b6..." style="width: 140px;">
      <label>Dataset:</label>
      <select id="dataset-select" style="min-width: 220px;"><option value="">Loading...</option></select>
    </div>
  </div>
  <div class="summary-bar" id="summary-bar"></div>
  <div class="player-bar">
    <button class="btn" id="btn-play">▶ Play</button>
    <button class="btn" id="btn-prev">◀ Step</button>
    <button class="btn" id="btn-next">Step ▶</button>
    <input type="number" id="frame-input" min="0" value="0" style="width: 55px;">
    <input type="range" id="frame-slider" class="slider" min="0" max="0" value="0">
    <span class="frame-badge" id="frame-badge">Frame 0 / 0</span>
  </div>
  <div class="main-grid">
    <div class="left-panel">
      <div class="plot-section">
        <div class="section-header">
          <span>3D Tracking Plots</span>
          <span style="font-size: 10px; color: #888;">Left: Window (t-1, t, t+1) | Right: Selected Cell Full Lineage</span>
        </div>
        <div class="plots-3d-row">
          <div id="plot-3d-window" class="plotly-box"></div>
          <div id="plot-3d-lineage" class="plotly-box"></div>
        </div>
      </div>
      <div class="plot-section">
        <div class="section-header">
          <span>2D MIP Projection Canvas (Overlay GT/Pred & 7μm Highlight Circle)</span>
          <button class="btn" id="btn-reset-2d" style="font-size: 10px; padding: 2px 6px;">Reset 2D Pan/Zoom</button>
        </div>
        <div class="mips-2d-row">
          <div class="mip-canvas-box">
            <div class="mip-title">XY Projection (Z-MIP)</div>
            <canvas id="canvas-xy"></canvas>
            <canvas class="gizmo-overlay" id="gizmo-xy"></canvas>
          </div>
          <div class="mip-canvas-box">
            <div class="mip-title">XZ Projection (Y-MIP)</div>
            <canvas id="canvas-xz"></canvas>
            <canvas class="gizmo-overlay" id="gizmo-xz"></canvas>
          </div>
          <div class="mip-canvas-box">
            <div class="mip-title">YZ Projection (X-MIP)</div>
            <canvas id="canvas-yz"></canvas>
            <canvas class="gizmo-overlay" id="gizmo-yz"></canvas>
          </div>
        </div>
      </div>
    </div>
    <div class="right-panel">
      <div class="inspector-card">
        <div class="section-header"><span>Cell Node Inspector</span></div>
        <table class="inspector-table">
          <tr><td class="label">Node ID</td><td class="val" id="insp-id">-</td></tr>
          <tr><td class="label">Status</td><td class="val" id="insp-status">-</td></tr>
          <tr><td class="label">Frame (t)</td><td class="val" id="insp-frame">-</td></tr>
          <tr><td class="label">Pos (Z, Y, X μm)</td><td class="val" id="insp-pos">-</td></tr>
          <tr><td class="label">Parent ID</td><td class="val" id="insp-parent">-</td></tr>
          <tr><td class="label">Matched Pred ID</td><td class="val" id="insp-pred">-</td></tr>
        </table>
      </div>
      <div class="inspector-card">
        <div class="section-header"><span>Consolidated RAW CSV (All Datasets)</span></div>
        <div style="font-size: 11px; margin-top: 5px;">
          <a id="link-csv-nodes" href="viewer_data/gt_nodes.csv" target="_blank" style="color: #4fc3f7; text-decoration: underline; display: block; margin-bottom: 6px;">gt_nodes.csv (All Dataset GT Nodes)</a>
          <a id="link-csv-edges" href="viewer_data/gt_edges.csv" target="_blank" style="color: #4fc3f7; text-decoration: underline; display: block;">gt_edges.csv (All Dataset GT Edges)</a>
        </div>
      </div>
    </div>
  </div>
  <script>
    let allDatasets = []; let currentPayload = null; let currentDatasetName = '';
    let currentFrame = 0; let maxFrames = 0; let isPlaying = false; let playTimer = null; let selectedNode = null;
    let zoomScale = 1.0; let panOffset = { x: 0, y: 0 }; let isDragging = false; let dragStart = { x: 0, y: 0 };

    async function init() {
      try {
        const res = await fetch('viewer_data/datasets.json');
        const data = await res.json();
        allDatasets = data.datasets || [];
        populateSelect(allDatasets);
        document.getElementById('dataset-search').addEventListener('input', (e) => filterDatasets(e.target.value));
        document.getElementById('dataset-select').addEventListener('change', (e) => loadDataset(e.target.value));
        setupPlayerControls(); setup2DCanvasEvents();
      } catch (e) { console.error('Failed to init datasets.json', e); }
    }

    function populateSelect(list) {
      const select = document.getElementById('dataset-select'); select.innerHTML = '';
      list.forEach((ds, idx) => {
        const opt = document.createElement('option'); opt.value = ds; opt.textContent = `No.${idx + 1}: ${ds}`; select.appendChild(opt);
      });
      if (list.length > 0) loadDataset(list[0]);
    }

    function filterDatasets(query) {
      const q = query.trim().toLowerCase(); if (!q) { populateSelect(allDatasets); return; }
      const noMatch = q.match(/^no\\.?\\s*(\\d+)$/);
      let filtered = [];
      if (noMatch) {
        const targetNo = parseInt(noMatch[1]);
        if (targetNo >= 1 && targetNo <= allDatasets.length) filtered = [allDatasets[targetNo - 1]];
      } else {
        filtered = allDatasets.filter(ds => ds.toLowerCase().includes(q));
      }
      populateSelect(filtered);
    }

    async function loadDataset(datasetName) {
      if (!datasetName) return; currentDatasetName = datasetName;
      try {
        const res = await fetch(`viewer_data/${datasetName}/viewer_data.json`);
        currentPayload = await res.json();
        renderSummaryCards(currentPayload);
        let maxT = 0;
        const allNodes = [...(currentPayload.gt_nodes_tp || []), ...(currentPayload.gt_nodes_fn || []), ...(currentPayload.pred_nodes_fp || [])];
        allNodes.forEach(n => { if (n[5] > maxT) maxT = n[5]; }); maxFrames = maxT;
        document.getElementById('frame-slider').max = maxT; document.getElementById('frame-input').max = maxT; currentFrame = 0;
        if (currentPayload.gt_nodes_tp && currentPayload.gt_nodes_tp.length > 0) {
          const n = currentPayload.gt_nodes_tp[0]; selectedNode = { z: n[0], y: n[1], x: n[2], gt_id: n[3], pred_id: n[4], t: n[5], parent_id: n[6], status: 'TP' };
        } else if (currentPayload.gt_nodes_fn && currentPayload.gt_nodes_fn.length > 0) {
          const n = currentPayload.gt_nodes_fn[0]; selectedNode = { z: n[0], y: n[1], x: n[2], gt_id: n[3], pred_id: n[4], t: n[5], parent_id: n[6], status: 'FN' };
        } else selectedNode = null;
        updateInspector(selectedNode); updateFrameView();
      } catch (e) { console.error(`Failed to load payload for ${datasetName}`, e); }
    }

    function renderSummaryCards(payload) {
      const summary = payload.global_summary || {}; const tooltips = payload.metadata?.tooltips || {};
      const bar = document.getElementById('summary-bar');
      bar.innerHTML = `
        <div class="card"><div class="card-title">Node Precision <span class="tooltip-icon" data-tooltip="検出ノードの正解率 (TP / (TP + FP))">i</span></div><div class="card-val">${(summary.node_precision || 0).toFixed(4)}</div></div>
        <div class="card"><div class="card-title">Node Recall <span class="tooltip-icon" data-tooltip="GT細胞の検出網羅率 (TP / (TP + FN))">i</span></div><div class="card-val">${(summary.node_recall || 0).toFixed(4)}</div></div>
        <div class="card"><div class="card-title">Node F1 Score <span class="tooltip-icon" data-tooltip="ノード検出調和平均 F1 スコア">i</span></div><div class="card-val">${(summary.node_f1 || 0).toFixed(4)}</div></div>
        <div class="card"><div class="card-title">Edge F1 Score <span class="tooltip-icon" data-tooltip="トラッキングリンク追跡一致率 F1 スコア">i</span></div><div class="card-val">${(summary.edge_f1 || 0).toFixed(4)}</div></div>
        <div class="card" style="border-left-color: #ab47bc;"><div class="card-title">Isolated GT Nodes <span class="tooltip-icon" data-tooltip="${tooltips.isolated_gt_nodes || '孤立GTノード'}">i</span></div><div class="card-val">${summary.isolated_gt_nodes_count || 0} / ${summary.gt_nodes_total || 0}</div></div>
        <div class="card" style="border-left-color: #2e7d32;"><div class="card-title">Node TP <span class="tooltip-icon" data-tooltip="${tooltips.gt_node_tp || 'TP'}">i</span></div><div class="card-val">${summary.node_tp_count || 0}</div></div>
        <div class="card" style="border-left-color: #c62828;"><div class="card-title">Node FN <span class="tooltip-icon" data-tooltip="${tooltips.gt_node_fn || 'FN'}">i</span></div><div class="card-val">${summary.node_fn_count || 0}</div></div>
        <div class="card" style="border-left-color: #ef6c00;"><div class="card-title">Node FP <span class="tooltip-icon" data-tooltip="${tooltips.pred_node_fp || 'FP'}">i</span></div><div class="card-val">${summary.node_fp_count || 0}</div></div>
        <div class="card" style="border-left-color: #ffb74d;"><div class="card-title">Pred Edge FP <span class="tooltip-icon" data-tooltip="${tooltips.pred_edges_fp || '誤予測エッジ'}">i</span></div><div class="card-val">${summary.pred_edges_fp_count || 0}</div></div>
      `;
    }

    function setupPlayerControls() {
      const btnPlay = document.getElementById('btn-play');
      btnPlay.addEventListener('click', () => {
        isPlaying = !isPlaying; btnPlay.textContent = isPlaying ? '❚❚ Pause' : '▶ Play';
        if (isPlaying) { playTimer = setInterval(() => { currentFrame = (currentFrame + 1) % (maxFrames + 1); updateFrameView(); }, 300); }
        else clearInterval(playTimer);
      });
      document.getElementById('btn-prev').addEventListener('click', () => setFrame(currentFrame - 1));
      document.getElementById('btn-next').addEventListener('click', () => setFrame(currentFrame + 1));
      document.getElementById('frame-slider').addEventListener('input', (e) => setFrame(parseInt(e.target.value)));
      document.getElementById('frame-input').addEventListener('change', (e) => setFrame(parseInt(e.target.value)));
    }

    function setFrame(f) { if (f < 0) f = 0; if (f > maxFrames) f = maxFrames; currentFrame = f; updateFrameView(); }
    function updateFrameView() {
      document.getElementById('frame-slider').value = currentFrame; document.getElementById('frame-input').value = currentFrame;
      document.getElementById('frame-badge').textContent = `Frame ${currentFrame} / ${maxFrames}`;
      render3DWindowPlot(); render3DLineagePlot(); render2DMIPCanvas(); drawGizmos();
    }

    function updateInspector(node) {
      if (!node) return;
      document.getElementById('insp-id').textContent = node.gt_id !== null ? node.gt_id : `Pred_${node.pred_id}`;
      document.getElementById('insp-status').innerHTML = `<span class="badge bg-${node.status.toLowerCase()}">${node.status}</span>`;
      document.getElementById('insp-frame').textContent = node.t;
      document.getElementById('insp-pos').textContent = `(${node.z.toFixed(2)}, ${node.y.toFixed(2)}, ${node.x.toFixed(2)})`;
      document.getElementById('insp-parent').textContent = node.parent_id !== null ? node.parent_id : '-';
      document.getElementById('insp-pred').textContent = node.pred_id !== null ? node.pred_id : '-';
    }

    function render3DWindowPlot() {
      if (!currentPayload) return;
      const targetFrames = [currentFrame - 1, currentFrame, currentFrame + 1];
      const filterNodes = (arr, status) => (arr || []).filter(n => targetFrames.includes(n[5])).map(n => ({ z: n[0], y: n[1], x: n[2], id: n[3], pred_id: n[4], t: n[5], parent_id: n[6], status }));
      const tpNodes = filterNodes(currentPayload.gt_nodes_tp, 'TP'); const fnNodes = filterNodes(currentPayload.gt_nodes_fn, 'FN'); const fpNodes = filterNodes(currentPayload.pred_nodes_fp, 'FP');
      const createTrace = (nodes, color, name) => ({
        x: nodes.map(n => n.x), y: nodes.map(n => n.y), z: nodes.map(n => n.z), mode: 'markers', type: 'scatter3d', name, marker: { size: 5, color },
        text: nodes.map(n => `ID: ${n.id}<br>t: ${n.t}<br>Pos: (${n.z.toFixed(1)}, ${n.y.toFixed(1)}, ${n.x.toFixed(1)})`), hoverinfo: 'text'
      });
      const data = [createTrace(tpNodes, '#2e7d32', 'GT TP'), createTrace(fnNodes, '#c62828', 'GT FN'), createTrace(fpNodes, '#ef6c00', 'Pred FP')];
      const layout = { margin: { l: 0, r: 0, b: 0, t: 0 }, paper_bgcolor: '#141414', plot_bgcolor: '#141414', scene: { xaxis: { title: 'X (μm)', color: '#888' }, yaxis: { title: 'Y (μm)', color: '#888' }, zaxis: { title: 'Z (μm)', color: '#888' }, bgcolor: '#141414' }, showlegend: true, legend: { x: 0, y: 1, font: { color: '#eee', size: 10 } } };
      Plotly.react('plot-3d-window', data, layout, { responsive: true });
    }

    function render3DLineagePlot() {
      if (!currentPayload || !selectedNode) return;
      const targetId = selectedNode.gt_id;
      const allNodes = [...(currentPayload.gt_nodes_tp || []), ...(currentPayload.gt_nodes_fn || [])];
      const lineageNodes = allNodes.filter(n => n[3] === targetId || n[6] === targetId).map(n => ({ z: n[0], y: n[1], x: n[2], id: n[3], t: n[5] }));
      lineageNodes.sort((a, b) => a.t - b.t);
      const trace = {
        x: lineageNodes.map(n => n.x), y: lineageNodes.map(n => n.y), z: lineageNodes.map(n => n.z), mode: 'lines+markers', type: 'scatter3d', name: `Track ${targetId}`,
        line: { color: '#4fc3f7', width: 4 }, marker: { size: 6, color: '#ab47bc' }, text: lineageNodes.map(n => `ID: ${n.id}<br>t: ${n.t}`), hoverinfo: 'text'
      };
      const layout = { margin: { l: 0, r: 0, b: 0, t: 0 }, paper_bgcolor: '#141414', plot_bgcolor: '#141414', scene: { xaxis: { title: 'X (μm)', color: '#888' }, yaxis: { title: 'Y (μm)', color: '#888' }, zaxis: { title: 'Z (μm)', color: '#888' }, bgcolor: '#141414' }, showlegend: false };
      Plotly.react('plot-3d-lineage', [trace], layout, { responsive: true });
    }

    function setup2DCanvasEvents() {
      ['xy', 'xz', 'yz'].forEach(type => {
        const canvas = document.getElementById(`canvas-${type}`);
        canvas.addEventListener('wheel', (e) => { e.preventDefault(); zoomScale *= (e.deltaY < 0 ? 1.1 : 0.9); render2DMIPCanvas(); });
        canvas.addEventListener('mousedown', (e) => { isDragging = true; dragStart = { x: e.clientX - panOffset.x, y: e.clientY - panOffset.y }; });
        window.addEventListener('mousemove', (e) => { if (!isDragging) return; panOffset.x = e.clientX - dragStart.x; panOffset.y = e.clientY - dragStart.y; render2DMIPCanvas(); });
        window.addEventListener('mouseup', () => { isDragging = false; });
      });
      document.getElementById('btn-reset-2d').addEventListener('click', () => { zoomScale = 1.0; panOffset = { x: 0, y: 0 }; render2DMIPCanvas(); });
    }

    function render2DMIPCanvas() {
      if (!currentDatasetName) return; const frameStr = String(currentFrame).padStart(3, '0');
      ['xy', 'xz', 'yz'].forEach(type => {
        const canvas = document.getElementById(`canvas-${type}`); const ctx = canvas.getContext('2d'); const img = new Image();
        img.src = `viewer_data/${currentDatasetName}/mips/frame_${frameStr}_${type}.png`;
        img.onload = () => {
          canvas.width = canvas.parentElement.clientWidth; canvas.height = canvas.parentElement.clientHeight;
          ctx.clearRect(0, 0, canvas.width, canvas.height); ctx.save(); ctx.translate(panOffset.x, panOffset.y); ctx.scale(zoomScale, zoomScale);
          ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
          if (currentPayload) {
            const frameNodes = [
              ...(currentPayload.gt_nodes_tp || []).filter(n => n[5] === currentFrame).map(n => ({ z: n[0], y: n[1], x: n[2], id: n[3], st: 'TP' })),
              ...(currentPayload.gt_nodes_fn || []).filter(n => n[5] === currentFrame).map(n => ({ z: n[0], y: n[1], x: n[2], id: n[3], st: 'FN' })),
              ...(currentPayload.pred_nodes_fp || []).filter(n => n[5] === currentFrame).map(n => ({ z: n[0], y: n[1], x: n[2], id: n[3], st: 'FP' }))
            ];
            frameNodes.forEach(n => {
              let px = 0, py = 0;
              if (type === 'xy') { px = (n.x / 100) * canvas.width; py = (n.y / 100) * canvas.height; }
              else if (type === 'xz') { px = (n.x / 100) * canvas.width; py = (n.z / 100) * canvas.height; }
              else if (type === 'yz') { px = (n.y / 100) * canvas.width; py = (n.z / 100) * canvas.height; }
              ctx.beginPath(); ctx.arc(px, py, 4, 0, 2 * Math.PI); ctx.fillStyle = n.st === 'TP' ? '#2e7d32' : n.st === 'FN' ? '#c62828' : '#ef6c00'; ctx.fill();
              if (selectedNode && selectedNode.gt_id === n.id) {
                ctx.beginPath(); ctx.arc(px, py, 14, 0, 2 * Math.PI); ctx.strokeStyle = '#4fc3f7'; ctx.lineWidth = 2; ctx.stroke();
              }
            });
          }
          ctx.restore();
        };
      });
    }

    function drawGizmos() {
      const gizmoConfigs = { 'xy': { labelX: 'X (R)', colorX: '#f44336', labelY: 'Y (D)', colorY: '#4caf50' }, 'xz': { labelX: 'X (R)', colorX: '#f44336', labelY: 'Z (D)', colorY: '#2196f3' }, 'yz': { labelX: 'Y (R)', colorX: '#4caf50', labelY: 'Z (D)', colorY: '#2196f3' } };
      ['xy', 'xz', 'yz'].forEach(type => {
        const c = document.getElementById(`gizmo-${type}`); const ctx = c.getContext('2d'); c.width = 45; c.height = 45; const cfg = gizmoConfigs[type];
        ctx.clearRect(0, 0, 45, 45); ctx.lineWidth = 2;
        ctx.beginPath(); ctx.moveTo(8, 37); ctx.lineTo(35, 37); ctx.strokeStyle = cfg.colorX; ctx.stroke(); ctx.fillStyle = cfg.colorX; ctx.font = '9px sans-serif'; ctx.fillText(cfg.labelX, 22, 33);
        ctx.beginPath(); ctx.moveTo(8, 37); ctx.lineTo(8, 10); ctx.strokeStyle = cfg.colorY; ctx.stroke(); ctx.fillStyle = cfg.colorY; ctx.fillText(cfg.labelY, 12, 18);
      });
    }

    init();
  </script>
</body>
</html>"""
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(advanced_html_content)
    print(f"Ensured advanced index.html template at {html_path}")
    
    # 4. GitHub Pages (gh-pages ブランチ) へ直接コミット＆プッシュデプロイ
    print("\n--- GitHub Pages (gh-pages) へのコミット & プッシュを実行中 ---")
    push_to_github_pages(WORKING_DIR, github_repo, branch_name, github_token, push_github)
    print("\n[完了] 高度 index.html および 統合 RAW CSV ファイルが GitHub Pages へ無事デプロイされました！")

# ==============================================================================
# 【手動実行用】実行する場合は以下のコメントアウトを解除してセルを実行してください
# ==============================================================================
# run_raw_csv_and_deploy_viewer(export_csv=True, push_github=True)
